In [12]:
#read in notes and noteStatusHistory files (downloaded from X on 11/14/24)

import pandas as pd

notes = pd.read_csv('~/Downloads/notes-00000.tsv', sep='\t')
status_history = pd.read_csv('~/Downloads/noteStatusHistory-00000.tsv', sep='\t')

/var/folders/z0/qzq8g2qx44lcdrv4wy390xdc0000gs/T/ipykernel_34224/820479913.py:5: DtypeWarning: Columns (5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  notes = pd.read_csv('~/Downloads/notes-00000.tsv', sep='\t')
/var/folders/z0/qzq8g2qx44lcdrv4wy390xdc0000gs/T/ipykernel_34224/820479913.py:6: DtypeWarning: Columns (10,19) have mixed types. Specify dtype option on import or set low_memory=False.
  status_history = pd.read_csv('~/Downloads/noteStatusHistory-00000.tsv', sep='\t')


In [54]:
# convert all numbers to strings, so that none of the numbers get rounded when I combine into one large df:

notes = notes.astype(str)
status_history = status_history.astype(str)

In [56]:
# dropping columns from Notes I don't need:

columns = "believable, harmful, validationDifficulty"
quoted_columns = [f'{col.strip()}' for col in columns.split(",")]
print(quoted_columns)

['believable', 'harmful', 'validationDifficulty']


In [66]:
notes_clean = notes.drop(columns=quoted_columns)

In [20]:
# Merge Notes and Status_History dfs
notesandstatus = notes_clean.merge(status_history, on='noteId', how='outer')

In [22]:
# Remove all unnecessary columns
columns_to_keep = [
    "noteId",
    "noteAuthorParticipantId_x",
    "createdAtMillis_x",
    "tweetId",
    "classification",
    "summary",
    "timestampMillisOfFirstNonNMRStatus",
    "firstNonNMRStatus"
]
notesandstatus = notesandstatus[columns_to_keep]
notesandstatus.to_csv('~/community_notes_data/notesandstatus.csv', index=False)

In [14]:
# print first 100 rows to see what I have - begin explorations

notes_100 = notesandstatus.head(100)
notes_100.to_csv('notes_100.csv', index=False)

In [16]:
unique_tweet_ids = notesandstatus['tweetId'].nunique()
print(f"Number of unique tweet IDs: {unique_tweet_ids}")


Number of unique tweet IDs: 864936


In [28]:
# load political keywords into a list
keywords = [
    "activist", "amendment", "anti-corruption", "assembly", "autocracy", "authoritarian", 
    "Ballot", "biden", "campaign", "candidate", "capitalist", "caucus", "censorship", 
    "city council", "climate policy", "Clinton", "civic", "civil liberties", "council", 
    "Congress", "Congressman", "congresswoman", "conservative", "constitution", "corruption", 
    "Covid", "debate", "Democrat", "democracy", "diplomacy", "election", "equality", 
    "European Union", "executive", "federal", "foreign", "freedom", "governor", "Hamas", 
    "Harris", "healthcare", "Hezbollah", "House of Representatives", "impeachment", "immigration", 
    "infrastructure", "Iran", "Israel", "judiciary", "justice", "Kim Jong Un", "law", "left-wing", 
    "legislation", "liberal", "lobbyist", "mandate", "mayor", "Mcconnell", "military", "Movement", 
    "MP", "nation", "national security", "nationalism", "NATO", "Netanyahu", "Obama", "PAC", 
    "parliament", "party", "patriot", "patriotism", "Pelosi", "polling", "policy", "poverty", 
    "power", "president", "progressive", "public office", "Putin", "rally", "reform", "regulation", 
    "relations", "representative", "Republican", "rights", "sanctions", "Schumer", "Senate", "senator", 
    "socialism", "socialist", "sovereignty", "state", "stimulus", "Supreme court", "Surveillance", 
    "starmer", "tariffs", "taxes", "transparency", "Trump", "Ukraine", "Vance", "veto", "vote", 
    "Walz", "welfare", "White house", "Xi JinPing", "Zelensky"
]


In [24]:
df = notesandstatus

In [30]:
# Check for keywords in "summary" column (text of Notes)

import re

# ensure all objects in summary column are strings
df['summary'] = df['summary'].astype(str)

# uses a regular expression to search for each keyword in the "summary" text
def find_keywords(text, keywords):
    found_keywords = [
        keyword for keyword in keywords 
        if re.search(rf'\b{re.escape(keyword)}\b', str(text), re.IGNORECASE)
    ]
    return found_keywords

# Apply find_keywords function to the 'summary' column and put resulting keywords into a new column "found_keywords"
df['found_keywords'] = df['summary'].apply(lambda x: find_keywords(x, keywords))

# Identify rows where keywords were found
political_notes = df[df['found_keywords'].str.len() > 0]

# save
political_notes.to_csv('~/community_notes_data/political_notes.csv', index=False)

In [32]:
political_notes_100 = political_notes.head(100)
political_notes_100.to_csv('political_notes_100.csv', index=False)

In [34]:
# randomize political_notes and manually check for accuracy (are they actually political?)
political_notes = pd.read_csv("/Users/ryanmurtfeldt/community_notes_data/political_notes.csv")
political_notes_rand = political_notes.sample(frac=1).reset_index(drop=True)

In [36]:
# manually checking 100 random notes to see if actually political
political_notes_rand_100 = political_notes_rand.head(100)
political_notes_rand_100.to_csv("political_notes_rand_100.csv", index=False)

In [97]:
# creating a nonpolitical df by pulling out (from df) all rows that meet two qualifications, 1) they don't contain any keywords and 
# 2) their tweetId doesn't exist in the political_notes df
# It seems implausible that there aren't ANY notes that meet these qualifications, but that's what my investigation below confirms.
# Creating a dataframe of all non-political tweets (based on NOT having any keywords in summary column)

# Extract the tweet IDs from the political_notes DataFrame
political_tweet_ids = set(political_notes['tweetId'])

# Filter rows where 'found_keywords' is empty and 'tweetId' is not in political_tweet_ids
nonpolitical_notes = df[
    (df['found_keywords'].str.len() == 0) & (~df['tweetId'].isin(political_tweet_ids))
]

nonpolitical_notes.to_csv('~/community_notes_data/nonpolitical_notes.csv', index=False)

In [99]:
# comparing the length of political and nonpolitical notes:
len(nonpolitical_notes)

1225952

In [81]:
len(political_notes)

340799

In [83]:
len(df)

1566751

In [117]:
# testing to make sure there aren't any tweetIds that exist in both dataframes:
common_tweet_ids = set(political_notes['tweetId']).intersection(nonpolitical_notes['tweetId'])
print(len(common_tweet_ids))

0


In [121]:
# grouping all rows (in political_notes) that share the same tweetId
# just to see how many there are and to help confirm the above code is working as intended
duplicate_tweet_ids = political_notes['tweetId'][political_notes['tweetId'].duplicated()]
duplicates_df = political_notes[political_notes['tweetId'].isin(duplicate_tweet_ids)]
sorted_political_duplicates = duplicates_df.sort_values(by='tweetId')
sorted_political_duplicates.head(100).to_csv("sorted_political_duplicates_100.csv", index=False)

In [123]:
# grouping all rows (in nonpolitical_notes) that share the same tweetId
duplicate_nontweet_ids = nonpolitical_notes['tweetId'][nonpolitical_notes['tweetId'].duplicated()]
duplicates_nondf = nonpolitical_notes[nonpolitical_notes['tweetId'].isin(duplicate_nontweet_ids)]
sorted_nonpolitical_duplicates = duplicates_nondf.sort_values(by='tweetId')
sorted_nonpolitical_duplicates.head(100).to_csv("sorted_nonpolitical_duplicates_100.csv", index=False)

In [ ]:
# Next, manually checking to see if this process accurately groups the political and nonpolitical

In [39]:
# randomize political_notes and manually check for accuracy (are they actually political?)
nonpolitical_notes_rand = nonpolitical_notes.sample(frac=1).reset_index(drop=True)

In [42]:
# need to remove rows with blank summary
nonpolitical_notes_rand = nonpolitical_notes_rand[
    nonpolitical_notes_rand['summary'].notna() & nonpolitical_notes_rand['summary'].str.strip().ne("")
]
nonpolitical_notes_rand.to_csv("~/community_notes_data/nonpolitical_notes_rand.csv", index=False)

In [44]:
# manually checking 100 random notes to see if actually non-political
nonpolitical_notes_rand_100 = nonpolitical_notes_rand.head(100)
nonpolitical_notes_rand_100.to_csv("nonpolitical_notes_rand_100.csv", index=False)

In [74]:
#remove non-english from the non-political rows (by definition, the political_notes will be english because I was searching for english keywords)
%pip install langdetect

  Using cached langdetect-1.0.9.tar.gz (981 kB)
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993224 sha256=029630bddbb9ed9c1cec080fcf959a9f9976812234da32e4faf0b8c71be5aea4
  Stored in directory: /Users/ryanmurtfeldt/Library/Caches/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect
Note: you may need to restart the kernel to use updated packages.


In [4]:
nonpolitical_notes_rand = pd.read_csv("/Users/ryanmurtfeldt/community_notes_data/nonpolitical_notes_rand.csv")

In [46]:
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

# Ensure consistent results from langdetect
DetectorFactory.seed = 0

def is_english(text):
    try:
        return detect(text) == "en"
    except LangDetectException:
        return False  # Handles cases where detection fails

# Filter rows where 'summary' is detected as English
nonpolitical_notes_rand_en = nonpolitical_notes_rand[
    nonpolitical_notes_rand['summary'].apply(is_english)
]
nonpolitical_notes_rand_en.to_csv("~/community_notes_data/nonpolitical_notes_rand_en.csv", index=False)

In [50]:
nonpolitical_notes_rand_en_100 = nonpolitical_notes_rand_en.head(100)
nonpolitical_notes_rand_en_100.to_csv("nonpolitical_notes_rand_en_100.csv", index=False)

In [ ]:
# Preprocessing Cells to Skip for Now - see below:

In [ ]:
# read in each notes_ratings file

import pandas as pd

notes_ratings0 = pd.read_csv('~/Downloads/ratings-00000.tsv', sep='\t')
notes_ratings1 = pd.read_csv('~/Downloads/ratings-00001.tsv', sep='\t')
notes_ratings2 = pd.read_csv('~/Downloads/ratings-00002.tsv', sep='\t')
notes_ratings3 = pd.read_csv('~/Downloads/ratings-00003.tsv', sep='\t')
notes_ratings4 = pd.read_csv('~/Downloads/ratings-00004.tsv', sep='\t')
notes_ratings5 = pd.read_csv('~/Downloads/ratings-00005.tsv', sep='\t')
notes_ratings6 = pd.read_csv('~/Downloads/ratings-00006.tsv', sep='\t')
notes_ratings7 = pd.read_csv('~/Downloads/ratings-00007.tsv', sep='\t')
notes_ratings8 = pd.read_csv('~/Downloads/ratings-00008.tsv', sep='\t')
notes_ratings9 = pd.read_csv('~/Downloads/ratings-00009.tsv', sep='\t')
notes_ratings10 = pd.read_csv('~/Downloads/ratings-00010.tsv', sep='\t')
notes_ratings11 = pd.read_csv('~/Downloads/ratings-00011.tsv', sep='\t')
notes_ratings12 = pd.read_csv('~/Downloads/ratings-00012.tsv', sep='\t')
notes_ratings13 = pd.read_csv('~/Downloads/ratings-00013.tsv', sep='\t')
notes_ratings14 = pd.read_csv('~/Downloads/ratings-00014.tsv', sep='\t')
notes_ratings15 = pd.read_csv('~/Downloads/ratings-00015.tsv', sep='\t')

In [ ]:
# This didn't work - still too big for my RAM
# Loop through the notes_ratings one at a time; later once I know which notes ratings I want to look at and use.
# pre-process to make smaller (remove columns not needed); merge into single csv
notes_ratings_list = [notes_ratings0, notes_ratings1, notes_ratings2, notes_ratings3, notes_ratings4, notes_ratings5, notes_ratings6, notes_ratings7, notes_ratings8, notes_ratings9, notes_ratings10, notes_ratings11, notes_ratings12, notes_ratings13, notes_ratings14, notes_ratings15]

# dropping columns from Notes_Ratings I don't need
# Initialize an empty dictionary to store the processed DataFrames
processed_files = {}

# Loop through the files and process them
for i, file in enumerate(notes_ratings_list):
    # Define columns to drop
    columns = "helpful, notHelpful, helpfulInformative, helpfulEmpathetic, helpfulUniqueContext, notHelpfulOpinionSpeculationOrBias, notHelpfulOutdated, notHelpfulOffTopic"
    quoted_columns = [col.strip() for col in columns.split(",")]
    
    # Process the file
    notes_ratings_clean = file.drop(columns=quoted_columns, errors='ignore')
    notes_ratings_clean = notes_ratings_clean.astype(str)
    
    # Save the processed DataFrame in the dictionary with a unique key
    processed_files[f'file_{i}'] = notes_ratings_clean

In [ ]:
# skip for now
# read in userEnrollment file when/if needed
import pandas as pd

enrollment = pd.read_csv('~/Downloads/userEnrollment-00000.tsv', sep='\t')